# Phase 03 — Psychological Capital → Employability

## Step 11 — Final Model Validation and Freeze

මෙම notebook එකේ අරමුණ Step 08 එකේ select කර save කරන ලද final Psychological Capital prediction model එක deployment සහ final research use සඳහා comprehensive ලෙස validate කර freeze කිරීමයි.

මෙම step එකේ model එක නැවත train හෝ tune නොකරයි.

Validation process එකෙන් පහත කරුණු පරීක්ෂා කරයි:

- final model artifact integrity
- model metadata consistency
- feature schema සහ feature-order consistency
- exact saved prediction reproducibility
- deterministic prediction behaviour
- valid සහ invalid input handling
- prediction range behaviour
- low, middle සහ high Psychological Capital boundary behaviour
- recommendation-system compatibility
- stored model-performance evidence consistency
- final artifact hashing සහ freeze-manifest creation

Model performance estimate සඳහා Step 08 nested cross-validation results භාවිත කරන අතර Step 11 එකේ training data මත නව accuracy claim එකක් generate නොකරයි.

Final validation සම්පූර්ණ වූ පසු exact saved model artifact එක frozen model ලෙස භාවිත කරනු ලබයි.

In [40]:
from pathlib import Path

import json
import hashlib
import shutil

import numpy as np
import pandas as pd
import joblib

In [41]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "phase3_psycap":
    PROJECT_DIR = PROJECT_DIR.parent.parent

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "phase3_psycap_prepared.csv"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "phase3"
)

MODEL_PATH = (
    MODEL_DIR
    / "phase3_psycap_selected_model.pkl"
)

METADATA_PATH = (
    MODEL_DIR
    / "phase3_psycap_model_metadata.json"
)

SCHEMA_PATH = (
    MODEL_DIR
    / "phase3_psycap_feature_schema.json"
)

REFERENCE_PATH = (
    MODEL_DIR
    / "phase3_psycap_reference_predictions.csv"
)

PARAMS_PATH = (
    MODEL_DIR
    / "phase3_tuned_hyperparameters.json"
)

REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "final_validation"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_DIR:", PROJECT_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("REPORT_DIR:", REPORT_DIR)

PROJECT_DIR: d:\internship-employability-project
MODEL_PATH: d:\internship-employability-project\models\phase3\phase3_psycap_selected_model.pkl
REPORT_DIR: d:\internship-employability-project\reports\phase3\final_validation


## Final Artifact Verification

Final model validation එක ආරම්භ කිරීමට පෙර Step 08 එකේ save කරන ලද model, metadata, feature schema, reference predictions සහ tuned-parameter files available ද යන්න පරීක්ෂා කරයි.

Missing artifact එකක් තිබේ නම් final freeze process එක continue නොකළ යුතුය.

In [42]:
required_artifacts = {
    "Final_Model": MODEL_PATH,
    "Model_Metadata": METADATA_PATH,
    "Feature_Schema": SCHEMA_PATH,
    "Reference_Predictions": REFERENCE_PATH,
    "Tuned_Hyperparameters": PARAMS_PATH
}

artifact_rows = []

for name, path in required_artifacts.items():

    artifact_rows.append({
        "Artifact": name,
        "Path": str(path),
        "Exists": path.exists()
    })

artifact_check_df = pd.DataFrame(
    artifact_rows
)

artifact_check_df

,Artifact,Path,Exists
0,Final_Model,d:\internship-employability-project\models\pha...,True
1,Model_Metadata,d:\internship-employability-project\models\pha...,True
2,Feature_Schema,d:\internship-employability-project\models\pha...,True
3,Reference_Predictions,d:\internship-employability-project\models\pha...,True
4,Tuned_Hyperparameters,d:\internship-employability-project\models\pha...,True


Stop if artifact missing

In [43]:
if not artifact_check_df["Exists"].all():

    missing = artifact_check_df.loc[
        ~artifact_check_df["Exists"],
        "Artifact"
    ].tolist()

    raise FileNotFoundError(
        f"Missing required final artifacts: {missing}"
    )

print(
    "All required final artifacts are available."
)

All required final artifacts are available.


Load exact frozen candidate

In [44]:
final_model = joblib.load(
    MODEL_PATH
)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    metadata = json.load(file)

with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as file:

    feature_schema = json.load(file)

with open(
    PARAMS_PATH,
    "r",
    encoding="utf-8"
) as file:

    tuned_parameters = json.load(file)

print(
    "Saved final model:",
    metadata["model_name"]
)

print(
    final_model
)

Saved final model: ElasticNet
Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 ElasticNet(alpha=np.float64(0.14873521072935117), l1_ratio=0.0,
                            max_iter=50000, random_state=42))])


Model Hash Integrity

In [45]:
def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        for block in iter(
            lambda: file.read(4096),
            b""
        ):
            sha256.update(block)

    return sha256.hexdigest()

In [46]:
current_model_hash = calculate_sha256(
    MODEL_PATH
)

saved_model_hash = metadata[
    "model_sha256"
]

hash_matches = (
    current_model_hash
    ==
    saved_model_hash
)

print(
    "Saved SHA256:",
    saved_model_hash
)

print(
    "Current SHA256:",
    current_model_hash
)

print(
    "Model integrity valid:",
    hash_matches
)

Saved SHA256: 6d8ed0af743a35f91dccf7f85630056d5a9afb551f2b887b3ff5f4daca5aa75b
Current SHA256: 6d8ed0af743a35f91dccf7f85630056d5a9afb551f2b887b3ff5f4daca5aa75b
Model integrity valid: True


## Feature Contract Validation

Final model එක input features හතරක් exact order එකෙන් expect කරයි:

1. Hope_avg
2. Optimism_avg
3. Resilience_avg
4. SelfEfficacy_avg

Feature-order mismatch deployment prediction errors ඇති කළ හැකි බැවින් schema එක strict ලෙස validate කරයි.

In [47]:
EXPECTED_FEATURES = [
    "Hope_avg",
    "Optimism_avg",
    "Resilience_avg",
    "SelfEfficacy_avg"
]

EXPECTED_TARGET = "EVS_avg"

schema_features = feature_schema[
    "features"
]

schema_target = feature_schema[
    "target"
]

feature_contract_valid = (
    schema_features
    ==
    EXPECTED_FEATURES
)

target_contract_valid = (
    schema_target
    ==
    EXPECTED_TARGET
)

print(
    "Feature contract valid:",
    feature_contract_valid
)

print(
    "Target contract valid:",
    target_contract_valid
)

Feature contract valid: True
Target contract valid: True


Validate Pipeline Structure

In [48]:
pipeline_steps = list(
    final_model.named_steps.keys()
)

print(
    "Pipeline steps:",
    pipeline_steps
)

pipeline_structure_valid = (
    pipeline_steps
    ==
    ["scaler", "model"]
)

model_class_name = (
    final_model
    .named_steps["model"]
    .__class__
    .__name__
)

scaler_class_name = (
    final_model
    .named_steps["scaler"]
    .__class__
    .__name__
)

print(
    "Scaler class:",
    scaler_class_name
)

print(
    "Model class:",
    model_class_name
)

Pipeline steps: ['scaler', 'model']
Scaler class: StandardScaler
Model class: ElasticNet


Validate exact model type

In [49]:
expected_model_name = "ElasticNet"

model_type_valid = (
    model_class_name
    ==
    expected_model_name
)

scaler_type_valid = (
    scaler_class_name
    ==
    "StandardScaler"
)

print(
    "Correct scaler:",
    scaler_type_valid
)

print(
    "Correct model:",
    model_type_valid
)

Correct scaler: True
Correct model: True


Validate final hyperparameters

In [50]:
final_model_params = (
    final_model
    .named_steps["model"]
    .get_params()
)

final_parameter_df = pd.DataFrame([
    {
        "Parameter": "alpha",
        "Value": final_model_params[
            "alpha"
        ]
    },
    {
        "Parameter": "l1_ratio",
        "Value": final_model_params[
            "l1_ratio"
        ]
    },
    {
        "Parameter": "max_iter",
        "Value": final_model_params[
            "max_iter"
        ]
    }
])

final_parameter_df

,Parameter,Value
0,alpha,0.148735
1,l1_ratio,0.000000
2,max_iter,50000.000000


### Final Regularization Interpretation

Final ElasticNet model එක `l1_ratio = 0.0` value එක select කර ඇත.

මෙය final optimization process එක sparse L1 feature elimination වෙනුවට L2-style shrinkage prefer කළ බව පෙන්වයි.

මෙය error එකක් නොවන අතර nested cross-validation evidence අනුව select කරන ලද optimized regularization behaviour එකයි.

Load validation data

In [51]:
df = pd.read_csv(
    DATA_PATH
)

X = df[
    EXPECTED_FEATURES
].copy()

y = df[
    EXPECTED_TARGET
].copy()

print(
    "Validation dataset shape:",
    df.shape
)

print(
    "Feature matrix shape:",
    X.shape
)

Validation dataset shape: (1000, 41)
Feature matrix shape: (1000, 4)


Data Contract Check

In [52]:
data_contract_checks = {
    "No_Missing_Features":
        not X.isnull().any().any(),

    "No_Missing_Target":
        not y.isnull().any(),

    "Correct_Feature_Count":
        X.shape[1] == 4,

    "Correct_Feature_Order":
        X.columns.tolist()
        ==
        EXPECTED_FEATURES,

    "All_Features_Within_1_5":
        bool(
            (
                (X >= 1)
                &
                (X <= 5)
            )
            .all()
            .all()
        )
}

data_contract_df = pd.DataFrame(
    list(
        data_contract_checks.items()
    ),
    columns=[
        "Check",
        "Status"
    ]
)

data_contract_df

,Check,Status
0,No_Missing_Features,True
1,No_Missing_Target,True
2,Correct_Feature_Count,True
3,Correct_Feature_Order,True
4,All_Features_Within_1_5,True


Reference Prediction Reproduction

In [53]:
reference_df = pd.read_csv(
    REFERENCE_PATH
)

reference_X = reference_df[
    EXPECTED_FEATURES
]

stored_reference_predictions = (
    reference_df[
        "Reference_Prediction"
    ]
    .to_numpy()
)

new_reference_predictions = (
    final_model.predict(
        reference_X
    )
)

reference_prediction_difference = np.abs(
    stored_reference_predictions
    -
    new_reference_predictions
)

max_reference_difference = float(
    reference_prediction_difference.max()
)

REFERENCE_ATOL = 1e-10
REFERENCE_RTOL = 1e-10

reference_predictions_identical = np.allclose(
    stored_reference_predictions,
    new_reference_predictions,
    atol=REFERENCE_ATOL,
    rtol=REFERENCE_RTOL
)

reference_difference_within_tolerance = (
    max_reference_difference
    <= REFERENCE_ATOL
)

print(
    "Reference predictions numerically identical:",
    reference_predictions_identical
)

print(
    "Maximum prediction difference:",
    f"{max_reference_difference:.18e}"
)

print(
    "Allowed absolute tolerance:",
    REFERENCE_ATOL
)

print(
    "Difference within tolerance:",
    reference_difference_within_tolerance
)

Reference predictions numerically identical: True
Maximum prediction difference: 4.440892098500626162e-16
Allowed absolute tolerance: 1e-10
Difference within tolerance: True


Determinism Validation

In [54]:
determinism_input = X.iloc[
    [0]
].copy()

determinism_predictions = []

for _ in range(100):

    determinism_predictions.append(
        float(
            final_model.predict(
                determinism_input
            )[0]
        )
    )

determinism_unique_values = (
    len(
        set(
            np.round(
                determinism_predictions,
                12
            )
        )
    )
)

determinism_valid = (
    determinism_unique_values == 1
)

print(
    "Unique predictions across 100 runs:",
    determinism_unique_values
)

print(
    "Deterministic behaviour:",
    determinism_valid
)

Unique predictions across 100 runs: 1
Deterministic behaviour: True


Strict Prediction Input Function

In [55]:
def validate_construct_input(
    input_data
):

    if not isinstance(
        input_data,
        dict
    ):
        raise ValueError(
            "Input must be a dictionary."
        )

    missing_features = [
        feature
        for feature in EXPECTED_FEATURES
        if feature not in input_data
    ]

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    unexpected_features = [
        feature
        for feature in input_data
        if feature not in EXPECTED_FEATURES
    ]

    if unexpected_features:
        raise ValueError(
            f"Unexpected features: {unexpected_features}"
        )

    for feature in EXPECTED_FEATURES:

        value = input_data[
            feature
        ]

        if not isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating
            )
        ):
            raise ValueError(
                f"{feature} must be numeric."
            )

        if value < 1 or value > 5:
            raise ValueError(
                f"{feature} must be between 1 and 5."
            )

    return True

Prediction wrapper

In [56]:
def safe_predict(
    input_data
):

    validate_construct_input(
        input_data
    )

    input_df = pd.DataFrame(
        [
            input_data
        ],
        columns=EXPECTED_FEATURES
    )

    prediction = float(
        final_model.predict(
            input_df
        )[0]
    )

    return prediction

Invalid Input Tests

In [57]:
invalid_test_results = []


test_cases = {

    "Missing_Feature": {
        "Hope_avg": 4,
        "Optimism_avg": 4,
        "Resilience_avg": 4
    },

    "Below_Range": {
        "Hope_avg": 0,
        "Optimism_avg": 4,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4
    },

    "Above_Range": {
        "Hope_avg": 4,
        "Optimism_avg": 6,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4
    },

    "Non_Numeric": {
        "Hope_avg": "high",
        "Optimism_avg": 4,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4
    },

    "Unexpected_Feature": {
        "Hope_avg": 4,
        "Optimism_avg": 4,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4,
        "Extra_Feature": 3
    }
}


for test_name, test_input in (
    test_cases.items()
):

    try:

        safe_predict(
            test_input
        )

        status = "FAIL"

    except ValueError:

        status = "PASS"

    invalid_test_results.append({
        "Test":
            test_name,

        "Status":
            status
    })


invalid_input_validation_df = (
    pd.DataFrame(
        invalid_test_results
    )
)

invalid_input_validation_df

,Test,Status
0,Missing_Feature,PASS
1,Below_Range,PASS
2,Above_Range,PASS
3,Non_Numeric,PASS
4,Unexpected_Feature,PASS


## Boundary Behaviour Validation

Final model එක Psychological Capital input range එකේ logical prediction behaviour පෙන්වනවද යන්න low, neutral/middle සහ high boundary profiles භාවිත කර පරීක්ෂා කරයි.

මෙම check එක model accuracy estimate එකක් නොව deployment sanity check එකකි.

In [58]:
boundary_inputs = {

    "Minimum_Profile": {
        feature: 1.0
        for feature
        in EXPECTED_FEATURES
    },

    "Middle_Profile": {
        feature: 3.0
        for feature
        in EXPECTED_FEATURES
    },

    "Maximum_Profile": {
        feature: 5.0
        for feature
        in EXPECTED_FEATURES
    }
}


boundary_rows = []


for case_name, case_input in (
    boundary_inputs.items()
):

    prediction = safe_predict(
        case_input
    )

    boundary_rows.append({
        "Case":
            case_name,

        "Prediction":
            prediction,

        "Within_Expected_Target_Range":
            1.0 <= prediction <= 5.0
    })


boundary_validation_df = pd.DataFrame(
    boundary_rows
)

boundary_validation_df

,Case,Prediction,Within_Expected_Target_Range
0,Minimum_Profile,1.851498,True
1,Middle_Profile,3.259239,True
2,Maximum_Profile,4.666980,True


Logical ordering

In [59]:
minimum_prediction = (
    boundary_validation_df.loc[
        boundary_validation_df[
            "Case"
        ]
        ==
        "Minimum_Profile",
        "Prediction"
    ]
    .iloc[0]
)

middle_prediction = (
    boundary_validation_df.loc[
        boundary_validation_df[
            "Case"
        ]
        ==
        "Middle_Profile",
        "Prediction"
    ]
    .iloc[0]
)

maximum_prediction = (
    boundary_validation_df.loc[
        boundary_validation_df[
            "Case"
        ]
        ==
        "Maximum_Profile",
        "Prediction"
    ]
    .iloc[0]
)


logical_boundary_order = (
    minimum_prediction
    <
    middle_prediction
    <
    maximum_prediction
)

print(
    "Logical boundary ordering:",
    logical_boundary_order
)

Logical boundary ordering: True


Full Prediction Range Check

In [60]:
full_predictions = (
    final_model.predict(
        X
    )
)

prediction_range_df = pd.DataFrame({
    "Metric": [
        "Minimum_Prediction",
        "Maximum_Prediction",
        "Mean_Prediction",
        "Predictions_Below_1",
        "Predictions_Above_5"
    ],

    "Value": [
        float(
            full_predictions.min()
        ),

        float(
            full_predictions.max()
        ),

        float(
            full_predictions.mean()
        ),

        int(
            np.sum(
                full_predictions < 1
            )
        ),

        int(
            np.sum(
                full_predictions > 5
            )
        )
    ]
})

prediction_range_df

,Metric,Value
0,Minimum_Prediction,2.086122
1,Maximum_Prediction,4.666980
2,Mean_Prediction,4.083333
3,Predictions_Below_1,0.000000
4,Predictions_Above_5,0.000000


Single-Feature Direction Sanity Test

In [61]:
direction_test_rows = []

BASE_VALUE = 3.0


for feature in EXPECTED_FEATURES:

    low_case = {
        name: BASE_VALUE
        for name
        in EXPECTED_FEATURES
    }

    high_case = (
        low_case.copy()
    )

    low_case[
        feature
    ] = 2.0

    high_case[
        feature
    ] = 4.0

    low_prediction = safe_predict(
        low_case
    )

    high_prediction = safe_predict(
        high_case
    )

    direction_test_rows.append({

        "Feature":
            feature,

        "Prediction_at_2":
            low_prediction,

        "Prediction_at_4":
            high_prediction,

        "Difference":
            high_prediction
            -
            low_prediction,

        "Expected_Positive_Direction":
            (
                high_prediction
                >
                low_prediction
            )
    })


direction_sanity_df = pd.DataFrame(
    direction_test_rows
)

direction_sanity_df

,Feature,Prediction_at_2,Prediction_at_4,Difference,Expected_Positive_Direction
0,Hope_avg,3.175914,3.342564,0.166650,True
1,Optimism_avg,3.156418,3.362060,0.205641,True
2,Resilience_avg,3.021209,3.497269,0.476060,True
3,SelfEfficacy_avg,2.979544,3.538934,0.559390,True


Exact Coefficient Direction Check

In [62]:
model_coefficients = (
    final_model
    .named_steps[
        "model"
    ]
    .coef_
)

coefficient_validation_df = pd.DataFrame({
    "Feature":
        EXPECTED_FEATURES,

    "Coefficient":
        model_coefficients,

    "Positive":
        model_coefficients > 0
})

coefficient_validation_df

,Feature,Coefficient,Positive
0,Hope_avg,0.033036,True
1,Optimism_avg,0.043687,True
2,Resilience_avg,0.098656,True
3,SelfEfficacy_avg,0.120245,True


Recommendation System Compatibility

In [63]:
RECOMMENDATION_REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "recommendations"
)

required_recommendation_files = [
    "Construct_Benchmarks.csv",
    "Step10_Validation.csv"
]

recommendation_compatibility_rows = []

for file_name in required_recommendation_files:

    file_path = (
        RECOMMENDATION_REPORT_DIR
        / file_name
    )

    recommendation_compatibility_rows.append({
        "Artifact":
            file_name,

        "Exists":
            file_path.exists()
    })


recommendation_compatibility_df = pd.DataFrame(
    recommendation_compatibility_rows
)

recommendation_compatibility_df

,Artifact,Exists
0,Construct_Benchmarks.csv,True
1,Step10_Validation.csv,True


Load Step 10 validation

In [64]:
STEP10_VALIDATION_PATH = (
    RECOMMENDATION_REPORT_DIR
    / "Step10_Validation.csv"
)

step10_validation_df = pd.read_csv(
    STEP10_VALIDATION_PATH
)

step10_validation_passed = (
    step10_validation_df[
        "Status"
    ]
    .eq(
        "PASS"
    )
    .all()
)

print(
    "Step 10 recommendation validation passed:",
    step10_validation_passed
)

Step 10 recommendation validation passed: True


## Final Performance Evidence Verification

Final model performance claim Step 08 nested cross-validation evidence මත පදනම් වේ.

Final model full approved dataset එක මත fit කර ඇති බැවින් Step 11 එකේ එම same dataset මත calculate කරන training R² final generalization accuracy ලෙස report නොකරයි.

Stored nested-CV metrics සහ metadata consistency පමණක් verify කරයි.

Extract stored model evidence

In [65]:
stored_performance_df = pd.DataFrame([
    {
        "Final_Model":
            metadata[
                "model_name"
            ],

        "Nested_CV_Mean_R2":
            metadata[
                "nested_cv_mean_r2"
            ],

        "Nested_CV_Std_R2":
            metadata[
                "nested_cv_std_r2"
            ],

        "Repeated_CV_Mean_R2":
            metadata[
                "repeated_cv_mean_r2"
            ],

        "Repeated_CV_Std_R2":
            metadata[
                "repeated_cv_std_r2"
            ],

        "CV_Mean_MAE":
            metadata[
                "cv_mean_mae"
            ],

        "CV_Mean_RMSE":
            metadata[
                "cv_mean_rmse"
            ],

        "Train_Validation_Gap":
            metadata[
                "train_validation_gap"
            ]
    }
])

stored_performance_df

,Final_Model,Nested_CV_Mean_R2,Nested_CV_Std_R2,Repeated_CV_Mean_R2,Repeated_CV_Std_R2,CV_Mean_MAE,CV_Mean_RMSE,Train_Validation_Gap
0,ElasticNet,0.573876,0.077173,0.574259,0.088289,0.136181,0.229558,0.021995


Freeze Artifact Hashes

In [66]:
freeze_artifacts = {
    "model":
        MODEL_PATH,

    "metadata":
        METADATA_PATH,

    "feature_schema":
        SCHEMA_PATH,

    "reference_predictions":
        REFERENCE_PATH,

    "tuned_hyperparameters":
        PARAMS_PATH
}


artifact_hashes = {}

for artifact_name, artifact_path in (
    freeze_artifacts.items()
):

    artifact_hashes[
        artifact_name
    ] = {
        "path":
            str(
                artifact_path
            ),

        "sha256":
            calculate_sha256(
                artifact_path
            )
    }


artifact_hashes

{'model': {'path': 'd:\\internship-employability-project\\models\\phase3\\phase3_psycap_selected_model.pkl',
  'sha256': '6d8ed0af743a35f91dccf7f85630056d5a9afb551f2b887b3ff5f4daca5aa75b'},
 'metadata': {'path': 'd:\\internship-employability-project\\models\\phase3\\phase3_psycap_model_metadata.json',
  'sha256': '6004ce747d809df9eabbb34155c883c823b0515e34d154ab318847dddd827907'},
 'feature_schema': {'path': 'd:\\internship-employability-project\\models\\phase3\\phase3_psycap_feature_schema.json',
  'sha256': '4668d129b04d180e33e566130d437de3075fbd2cd96b7fb89b66f8acc77aa4ca'},
 'reference_predictions': {'path': 'd:\\internship-employability-project\\models\\phase3\\phase3_psycap_reference_predictions.csv',
  'sha256': '4709e6eeb259f19c4d4ff287e98c38b73a145cc7da71d2457c1faccda06340fd'},
 'tuned_hyperparameters': {'path': 'd:\\internship-employability-project\\models\\phase3\\phase3_tuned_hyperparameters.json',
  'sha256': '9280199096ff1db39bf2e837e2d4e9f324d5b52575081b14022a4ec8ab3fb91c

Create Freeze Manifest

In [67]:
FREEZE_MANIFEST_PATH = (
    MODEL_DIR
    / "phase3_psycap_freeze_manifest.json"
)


freeze_manifest = {

    "phase":
        "Phase 03",

    "component":
        "Psychological Capital -> Employability",

    "final_model":
        metadata[
            "model_name"
        ],

    "model_artifact":
        MODEL_PATH.name,

    "feature_order":
        EXPECTED_FEATURES,

    "target":
        EXPECTED_TARGET,

    "model_status":
        "FROZEN",

    "retraining_allowed":
        False,

    "nested_cv_mean_r2":
        metadata[
            "nested_cv_mean_r2"
        ],

    "nested_cv_std_r2":
        metadata[
            "nested_cv_std_r2"
        ],

    "repeated_cv_mean_r2":
        metadata[
            "repeated_cv_mean_r2"
        ],

    "train_validation_gap":
        metadata[
            "train_validation_gap"
        ],

    "artifacts":
        artifact_hashes,

    "deployment_rule":
        (
            "Load the exact frozen model artifact. "
            "Do not refit, retrain, rescale, or change feature order."
        )
}


with open(
    FREEZE_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        freeze_manifest,
        file,
        indent=4
    )


print(
    "Freeze manifest saved:",
    FREEZE_MANIFEST_PATH
)

Freeze manifest saved: d:\internship-employability-project\models\phase3\phase3_psycap_freeze_manifest.json


Freeze Manifest Verification

In [68]:
with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:

    reloaded_manifest = json.load(
        file
    )

freeze_status_valid = (
    reloaded_manifest[
        "model_status"
    ]
    ==
    "FROZEN"
)

retraining_disabled = (
    reloaded_manifest[
        "retraining_allowed"
    ]
    is False
)

print(
    "Freeze status valid:",
    freeze_status_valid
)

print(
    "Retraining disabled:",
    retraining_disabled
)

Freeze status valid: True
Retraining disabled: True


In [69]:
all_artifacts_exist = bool(
    artifact_check_df[
        "Exists"
    ]
    .all()
)

all_invalid_inputs_rejected = bool(
    invalid_input_validation_df[
        "Status"
    ]
    .eq(
        "PASS"
    )
    .all()
)

all_boundary_predictions_valid = bool(
    boundary_validation_df[
        "Within_Expected_Target_Range"
    ]
    .all()
)

all_directions_valid = bool(
    direction_sanity_df[
        "Expected_Positive_Direction"
    ]
    .all()
)

dataset_predictions_in_range = bool(
    (
        full_predictions >= 1
    ).all()
    and
    (
        full_predictions <= 5
    ).all()
)

recommendation_artifacts_valid = bool(
    recommendation_compatibility_df[
        "Exists"
    ]
    .all()
)


step11_validation = pd.DataFrame({

    "Check": [

        "All final artifacts exist",

        "Saved model SHA256 matches metadata",

        "Final model loaded without retraining",

        "Feature contract matches expected schema",

        "Target contract matches expected schema",

        "Pipeline structure verified",

        "StandardScaler verified",

        "ElasticNet model verified",

        "Final hyperparameters inspected",

        "Dataset input contract valid",

        "Reference predictions reproduced within numerical tolerance",

        "Maximum reference prediction difference within tolerance",

        "Prediction determinism verified",

        "Invalid deployment inputs rejected",

        "Boundary predictions remain within target range",

        "Boundary predictions follow logical ordering",

        "Dataset predictions remain within target range",

        "Positive coefficient directions verified",

        "Feature-direction sanity tests passed",

        "Recommendation artifacts compatible",

        "Step 10 recommendation validation passed",

        "Stored nested-CV evidence available",

        "Freeze manifest created",

        "Freeze status verified",

        "Retraining disabled in manifest"
    ],

    "Status": [

        "PASS"
        if all_artifacts_exist
        else "FAIL",

        "PASS"
        if hash_matches
        else "FAIL",

        "PASS",

        "PASS"
        if feature_contract_valid
        else "FAIL",

        "PASS"
        if target_contract_valid
        else "FAIL",

        "PASS"
        if pipeline_structure_valid
        else "FAIL",

        "PASS"
        if scaler_type_valid
        else "FAIL",

        "PASS"
        if model_type_valid
        else "FAIL",

        "PASS",

        "PASS"
        if all(
            data_contract_checks.values()
        )
        else "FAIL",

       "PASS"
if reference_predictions_identical
else "FAIL",

"PASS"
if reference_difference_within_tolerance
else "FAIL",

        "PASS"
        if determinism_valid
        else "FAIL",

        "PASS"
        if all_invalid_inputs_rejected
        else "FAIL",

        "PASS"
        if all_boundary_predictions_valid
        else "FAIL",

        "PASS"
        if logical_boundary_order
        else "FAIL",

        "PASS"
        if dataset_predictions_in_range
        else "REVIEW",

        "PASS"
        if coefficient_validation_df[
            "Positive"
        ].all()
        else "REVIEW",

        "PASS"
        if all_directions_valid
        else "FAIL",

        "PASS"
        if recommendation_artifacts_valid
        else "FAIL",

        "PASS"
        if step10_validation_passed
        else "FAIL",

        "PASS",

        "PASS"
        if FREEZE_MANIFEST_PATH.exists()
        else "FAIL",

        "PASS"
        if freeze_status_valid
        else "FAIL",

        "PASS"
        if retraining_disabled
        else "FAIL"
    ]
})

step11_validation

,Check,Status
0,All final artifacts exist,PASS
1,Saved model SHA256 matches metadata,PASS
2,Final model loaded without retraining,PASS
3,Feature contract matches expected schema,PASS
4,Target contract matches expected schema,PASS
5,Pipeline structure verified,PASS
6,StandardScaler verified,PASS
7,ElasticNet model verified,PASS
8,Final hyperparameters inspected,PASS
9,Dataset input contract valid,PASS


Overall Freeze Decision

In [70]:
all_step11_checks_pass = (
    step11_validation[
        "Status"
    ]
    .eq(
        "PASS"
    )
    .all()
)


if all_step11_checks_pass:

    FINAL_STATUS = (
        "FROZEN_AND_VALIDATED"
    )

else:

    FINAL_STATUS = (
        "REVIEW_REQUIRED"
    )


print(
    "FINAL STEP 11 STATUS:",
    FINAL_STATUS
)

FINAL STEP 11 STATUS: FROZEN_AND_VALIDATED


Save all validation reports

In [71]:
artifact_check_df.to_csv(
    REPORT_DIR
    / "Artifact_Check.csv",
    index=False
)

final_parameter_df.to_csv(
    REPORT_DIR
    / "Final_Model_Parameters.csv",
    index=False
)

data_contract_df.to_csv(
    REPORT_DIR
    / "Data_Contract_Validation.csv",
    index=False
)

invalid_input_validation_df.to_csv(
    REPORT_DIR
    / "Invalid_Input_Validation.csv",
    index=False
)

boundary_validation_df.to_csv(
    REPORT_DIR
    / "Boundary_Validation.csv",
    index=False
)

prediction_range_df.to_csv(
    REPORT_DIR
    / "Prediction_Range_Validation.csv",
    index=False
)

direction_sanity_df.to_csv(
    REPORT_DIR
    / "Feature_Direction_Sanity.csv",
    index=False
)

coefficient_validation_df.to_csv(
    REPORT_DIR
    / "Coefficient_Validation.csv",
    index=False
)

recommendation_compatibility_df.to_csv(
    REPORT_DIR
    / "Recommendation_Compatibility.csv",
    index=False
)

stored_performance_df.to_csv(
    REPORT_DIR
    / "Final_Performance_Evidence.csv",
    index=False
)

step11_validation.to_csv(
    REPORT_DIR
    / "Step11_Final_Validation.csv",
    index=False
)

print(
    "Step 11 reports saved successfully."
)

Step 11 reports saved successfully.


Save final status

In [72]:
final_status_record = {

    "phase":
        "Phase 03",

    "final_model":
        metadata[
            "model_name"
        ],

    "status":
        FINAL_STATUS,

    "model_file":
        str(
            MODEL_PATH
        ),

    "freeze_manifest":
        str(
            FREEZE_MANIFEST_PATH
        ),

    "nested_cv_mean_r2":
        metadata[
            "nested_cv_mean_r2"
        ],

    "retraining_allowed":
        False
}


FINAL_STATUS_PATH = (
    MODEL_DIR
    / "phase3_psycap_final_status.json"
)


with open(
    FINAL_STATUS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_status_record,
        file,
        indent=4
    )


print(
    "Final status saved:",
    FINAL_STATUS_PATH
)

Final status saved: d:\internship-employability-project\models\phase3\phase3_psycap_final_status.json


In [73]:
print(
    "PHASE 03 — STEP 11 FINAL MODEL VALIDATION AND FREEZE"
)

print(
    "\nFINAL MODEL:"
)

print(
    metadata[
        "model_name"
    ]
)

print(
    "\nPERFORMANCE EVIDENCE:"
)

print(
    stored_performance_df
    .to_string(
        index=False
    )
)

print(
    "\nBOUNDARY VALIDATION:"
)

print(
    boundary_validation_df
    .to_string(
        index=False
    )
)

print(
    "\nVALIDATION MATRIX:"
)

print(
    step11_validation
    .to_string(
        index=False
    )
)

print(
    "\nFINAL STATUS:"
)

print(
    FINAL_STATUS
)

PHASE 03 — STEP 11 FINAL MODEL VALIDATION AND FREEZE

FINAL MODEL:
ElasticNet

PERFORMANCE EVIDENCE:
Final_Model  Nested_CV_Mean_R2  Nested_CV_Std_R2  Repeated_CV_Mean_R2  Repeated_CV_Std_R2  CV_Mean_MAE  CV_Mean_RMSE  Train_Validation_Gap
 ElasticNet           0.573876          0.077173             0.574259            0.088289     0.136181      0.229558              0.021995

BOUNDARY VALIDATION:
           Case  Prediction  Within_Expected_Target_Range
Minimum_Profile    1.851498                          True
 Middle_Profile    3.259239                          True
Maximum_Profile    4.666980                          True

VALIDATION MATRIX:
                                                      Check Status
                                  All final artifacts exist   PASS
                        Saved model SHA256 matches metadata   PASS
                      Final model loaded without retraining   PASS
                   Feature contract matches expected schema   PASS
            

## Step 11 අවසාන නිගමනය

Step 08 එකේ nested cross-validation evidence මත select කරන ලද final ElasticNet model එක comprehensive validation process එකකට යොමු කරන ලදී.

Final model artifact hash verification මගින් model file integrity confirm කරන ලදී. Feature schema, target contract, feature order, preprocessing pipeline සහ final model type validate කරන ලදී.

Saved reference predictions exact ලෙස reproduce වන බව verify කිරීමෙන් Step 08 සහ Step 11 අතර model behaviour වෙනස් වී නොමැති බව තහවුරු කරන ලදී.

Repeated identical predictions මගින් deterministic model behaviour validate කරන ලදී. Missing, invalid, out-of-range සහ unexpected input fields reject කරන strict deployment-input validation ද පරීක්ෂා කරන ලදී.

Minimum, middle සහ maximum Psychological Capital profiles භාවිත කර boundary behaviour validate කරන ලද අතර predictions logical order එකක් පෙන්වන බව පරීක්ෂා කරන ලදී.

Feature-direction sanity testing මගින් Step 09 explainability findings සහ final model prediction behaviour අතර consistency validate කරන ලදී.

Step 10 recommendation-system compatibility සහ user-facing recommendation validation ද final system validation process එකට ඇතුළත් කරන ලදී.

Final model generalization performance සඳහා Step 08 nested cross-validation evidence භාවිත කරන අතර full training dataset performance final accuracy ලෙස report නොකරයි.

සියලු final model artifacts සඳහා cryptographic hashes record කර freeze manifest එකක් සකස් කරන ලදී.

Final model freeze කිරීමෙන් පසු subsequent revalidation සහ deployment processes වල exact saved model artifact එක පමණක් භාවිත කරන අතර model retraining, refitting, rescaling හෝ feature-order වෙනස් කිරීම සිදු නොකරයි.

In [74]:
non_pass_checks = step11_validation[
    step11_validation["Status"] != "PASS"
].copy()

print(
    "Number of non-PASS checks:",
    len(non_pass_checks)
)

non_pass_checks

Number of non-PASS checks: 0


,Check,Status


In [75]:
print(
    step11_validation.to_string(
        index=False
    )
)

                                                      Check Status
                                  All final artifacts exist   PASS
                        Saved model SHA256 matches metadata   PASS
                      Final model loaded without retraining   PASS
                   Feature contract matches expected schema   PASS
                    Target contract matches expected schema   PASS
                                Pipeline structure verified   PASS
                                    StandardScaler verified   PASS
                                  ElasticNet model verified   PASS
                            Final hyperparameters inspected   PASS
                               Dataset input contract valid   PASS
Reference predictions reproduced within numerical tolerance   PASS
   Maximum reference prediction difference within tolerance   PASS
                            Prediction determinism verified   PASS
                         Invalid deployment inputs rejected   

In [76]:
for _, row in non_pass_checks.iterrows():
    print(
        f"{row['Check']} --> {row['Status']}"
    )

In [77]:
print(
    "Maximum reference prediction difference:",
    repr(max_reference_difference)
)

print(
    "Scientific notation:",
    f"{max_reference_difference:.18e}"
)

Maximum reference prediction difference: 4.440892098500626e-16
Scientific notation: 4.440892098500626162e-16


In [78]:
step11_validation[
    step11_validation["Status"] != "PASS"
]

,Check,Status


In [79]:
print(
    "FINAL STEP 11 STATUS:",
    FINAL_STATUS
)

FINAL STEP 11 STATUS: FROZEN_AND_VALIDATED
